# OMOP DE-SynPUF in BigQuery — a guided demo

This notebook works the **CMS DE-SynPUF 100k** dataset (synthetic Medicare
claims mapped to the **OMOP Common Data Model**) that `staging/` loaded into
BigQuery. It shows the query patterns you'll reuse against the real *All of Us*
CDR — same table shapes, same joins to the vocabulary.

The centrepiece is **concept-set expansion with `concept_ancestor`**: instead of
hand-picking one diagnosis code, we take the SNOMED concept for *Type 2 diabetes
mellitus* and pull its entire descendant hierarchy — a clinically coherent set —
then build and characterize a patient cohort from it.

**Prerequisite:** the `synpuf_omop` dataset is loaded (see `staging/README.md`).
If yours has a different name or project, edit the config cell below.

## 1. Setup

Two ways to run SQL in this notebook, and we use both:

- **`%%bigquery` cell magic** — for clean, SQL-only exploration. We set a
  *default dataset* so these cells can write `FROM person` instead of the full
  `` `project.synpuf_omop.person` ``.
- **`run(sql)` helper** — returns a pandas DataFrame, for when we parameterize
  the query or feed the result into a chart.

In [ ]:
# to_dataframe() needs db-dtypes for DATE/NUMERIC columns; install if missing.
try:
    import db_dtypes  # noqa: F401
except ImportError:
    !pip install -q db-dtypes

import os
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import bigquery

%matplotlib inline

# --- config: override here if your dataset/project differ ---
PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT")
DATASET = "synpuf_omop"
assert PROJECT, "GOOGLE_CLOUD_PROJECT is not set -- set PROJECT = '...' manually."

# No explicit location on the client: BigQuery routes each query to the region
# of the tables it references, so this works whether synpuf_omop is in the US
# multi-region or a single region like us-central1. (Pinning the wrong location
# is exactly what made `bq load` report 'dataset not found' during staging.)
client = bigquery.Client(project=PROJECT)

def run(sql: str) -> pd.DataFrame:
    """Run SQL, return a DataFrame."""
    return client.query(sql).to_dataframe()

def t(name: str) -> str:
    """Backtick-quoted fully-qualified table ref, for f-string SQL."""
    return f"`{PROJECT}.{DATASET}.{name}`"

print("Project:", PROJECT)
print("Dataset:", DATASET)

### Register the `%%bigquery` magic

The `%%bigquery` cell magic lives in a separate `bigquery-magics` package on
newer `google-cloud-bigquery`, and inside `google.cloud.bigquery` on older
ones. This cell loads whichever is present (installing `bigquery-magics` if
neither is), then points it at our project and default dataset. If it can't
load, only the one magic cell below is affected — everything else uses
`run(...)`.

In [ ]:
ip = get_ipython()

def _load_bq_magics():
    for ext in ("bigquery_magics", "google.cloud.bigquery"):
        try:
            ip.run_line_magic("load_ext", ext)
            return ext
        except Exception:
            pass
    return None

_ext = _load_bq_magics()
if _ext is None:
    !pip install -q bigquery-magics
    _ext = _load_bq_magics()

if _ext is None:
    print("WARNING: could not load the %%bigquery magic. The single magic cell "
          "below will error -- rewrite it with run(...) like the other cells.")
else:
    if _ext == "bigquery_magics":
        import bigquery_magics as _bqm
    else:
        from google.cloud.bigquery import magics as _bqm
    # Use our project and let magic cells reference tables unqualified.
    _bqm.context.project = PROJECT
    _bqm.context.default_query_job_config = bigquery.QueryJobConfig(
        default_dataset=f"{PROJECT}.{DATASET}"
    )
    print("%%bigquery magic loaded via:", _ext)

## 2. What's in here?

Row counts per table — a quick confirmation the load worked (you should see
`person` at 100,000).

In [ ]:
run(f"""
SELECT table_id AS table_name, row_count
FROM {t('__TABLES__')}
ORDER BY row_count DESC
""")

## 3. Who are these patients?

Every coded value in OMOP is a `concept_id` that joins to the **`concept`**
table for a human-readable name. Here's the sex breakdown, resolved to names —
note the unqualified table names, courtesy of the default dataset we set above.

In [ ]:
%%bigquery
SELECT c.concept_name AS sex, COUNT(*) AS n_persons
FROM person p
JOIN concept c ON c.concept_id = p.gender_concept_id
GROUP BY sex
ORDER BY n_persons DESC

SynPUF only carries `year_of_birth`, and the synthetic claims sit around
2008–2010, so we index age at **2010** purely for illustration.

In [ ]:
INDEX_YEAR = 2010  # synthetic data is ~2008-2010; index age here for illustration

ages = run(f"""
SELECT {INDEX_YEAR} - year_of_birth AS age
FROM {t('person')}
WHERE year_of_birth IS NOT NULL
""")

ax = ages["age"].plot(kind="hist", bins=30, edgecolor="white")
ax.set_title(f"Age distribution of all persons (indexed at {INDEX_YEAR})")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Persons")
plt.show()

print(f"{len(ages):,} persons, median age {ages['age'].median():.0f}")

## 4. Find the concept: *Type 2 diabetes mellitus*

Before we can expand a hierarchy we need its root **standard** concept. Rather
than hardcode an id, we look it up by name — the standard OMOP condition
vocabulary is **SNOMED**, and `standard_concept = 'S'` marks the concept you're
allowed to store in `condition_occurrence.condition_concept_id`.

In [ ]:
t2dm = run(f"""
SELECT concept_id, concept_name, vocabulary_id, domain_id,
       concept_class_id, standard_concept
FROM {t('concept')}
WHERE LOWER(concept_name) = 'type 2 diabetes mellitus'
  AND vocabulary_id = 'SNOMED'
  AND standard_concept = 'S'
""")
assert len(t2dm) > 0, "Concept not found -- check the vocabulary loaded."
t2dm

In [ ]:
T2DM_ID = int(t2dm.loc[0, "concept_id"])
print("Type 2 diabetes mellitus -> concept_id", T2DM_ID)

## 5. Expand the concept set with `concept_ancestor`

`concept_ancestor` is a pre-computed transitive closure of the vocabulary
hierarchy: one row for every (ancestor, descendant) pair. Asking for every
**descendant** of Type 2 diabetes gives us the whole coherent family of more
specific diagnoses — "Type 2 diabetes mellitus with renal complications",
"...with neuropathy", and so on — without naming any of them by hand.

This is *the* pattern for defining a phenotype from the hierarchy instead of a
brittle hand-typed code list.

In [ ]:
descendants = run(f"""
SELECT ca.descendant_concept_id AS concept_id, c.concept_name
FROM {t('concept_ancestor')} ca
JOIN {t('concept')} c ON c.concept_id = ca.descendant_concept_id
WHERE ca.ancestor_concept_id = {T2DM_ID}
ORDER BY c.concept_name
""")

print(f"{len(descendants):,} concepts in the Type 2 diabetes set (incl. itself)")
descendants.head(20)

## 6. Build the cohort

A patient is "in the cohort" if they have at least one `condition_occurrence`
whose `condition_concept_id` falls in that expanded set. We join
`condition_occurrence` straight to `concept_ancestor` — no need to
materialize the id list.

In [ ]:
cohort = run(f"""
WITH t2dm_set AS (
  SELECT descendant_concept_id AS concept_id
  FROM {t('concept_ancestor')}
  WHERE ancestor_concept_id = {T2DM_ID}
)
SELECT COUNT(DISTINCT co.person_id) AS n_patients
FROM {t('condition_occurrence')} co
JOIN t2dm_set s ON s.concept_id = co.condition_concept_id
""")

n_dm = int(cohort.loc[0, "n_patients"])
n_total = int(run(f"SELECT COUNT(*) AS n FROM {t('person')}").loc[0, "n"])
print(f"{n_dm:,} of {n_total:,} persons "
      f"({100 * n_dm / n_total:.1f}%) have a Type 2 diabetes condition")

## 7. Characterize the cohort

**Do diabetics skew older?** Overlay the cohort's age distribution on the
overall population (densities, so the smaller cohort is comparable).

In [ ]:
cohort_ages = run(f"""
WITH t2dm_set AS (
  SELECT descendant_concept_id AS concept_id
  FROM {t('concept_ancestor')} WHERE ancestor_concept_id = {T2DM_ID}
)
SELECT DISTINCT p.person_id, {INDEX_YEAR} - p.year_of_birth AS age
FROM {t('person')} p
JOIN {t('condition_occurrence')} co ON co.person_id = p.person_id
JOIN t2dm_set s ON s.concept_id = co.condition_concept_id
WHERE p.year_of_birth IS NOT NULL
""")

plt.hist(ages["age"], bins=30, alpha=0.5, density=True, label="All persons")
plt.hist(cohort_ages["age"], bins=30, alpha=0.5, density=True, label="T2DM cohort")
plt.legend()
plt.title("Age: Type 2 diabetes cohort vs. all persons")
plt.xlabel("Age (years)")
plt.ylabel("Density")
plt.show()

print(f"Median age -- all: {ages['age'].median():.0f},  "
      f"cohort: {cohort_ages['age'].median():.0f}")

**What else are they diagnosed with?** Top conditions co-occurring in the
cohort, excluding the diabetes set itself and unmapped (`concept_id = 0`)
records.

In [ ]:
cooccur = run(f"""
WITH t2dm_set AS (
  SELECT descendant_concept_id AS concept_id
  FROM {t('concept_ancestor')} WHERE ancestor_concept_id = {T2DM_ID}
),
cohort AS (
  SELECT DISTINCT co.person_id
  FROM {t('condition_occurrence')} co
  JOIN t2dm_set s ON s.concept_id = co.condition_concept_id
)
SELECT c.concept_name AS condition,
       COUNT(DISTINCT co.person_id) AS n_patients
FROM {t('condition_occurrence')} co
JOIN cohort ch ON ch.person_id = co.person_id
JOIN {t('concept')} c ON c.concept_id = co.condition_concept_id
WHERE co.condition_concept_id NOT IN (SELECT concept_id FROM t2dm_set)
  AND co.condition_concept_id != 0
GROUP BY condition
ORDER BY n_patients DESC
LIMIT 15
""")

ax = (cooccur.set_index("condition")["n_patients"]
             .sort_values()
             .plot(kind="barh"))
ax.set_title("Top conditions co-occurring with Type 2 diabetes")
ax.set_xlabel("Patients in cohort")
ax.set_ylabel("")
plt.tight_layout()
plt.show()
cooccur

## 8. Where to go next

You now have the core moves: resolve codes through **`concept`**, expand a
phenotype through **`concept_ancestor`**, and assemble a cohort by joining that
set to a clinical event table.

To extend this:

- **Drugs / measurements / procedures** follow the identical pattern — swap
  `condition_occurrence` for `drug_exposure`, `measurement`, or
  `procedure_occurrence` and pick a domain-appropriate ancestor concept.
- **Eras** (`condition_era`, `drug_era`) collapse repeated events into
  continuous periods — handy for "time on drug" style questions.
- **Vocabulary dates are strings.** We loaded `valid_start_date` /
  `valid_end_date` as `STRING` (Athena stores them `YYYYMMDD`), so parse on
  demand: `PARSE_DATE('%Y%m%d', valid_start_date)`.

Remember this is **synthetic** data — the distributions are for learning the
mechanics, not for drawing clinical conclusions.